In [4]:
!git clone https://github.com/meta-compute/sem-fracture-deep-learning.git

Cloning into 'sem-fracture-deep-learning'...
remote: Enumerating objects: 2691, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 2691 (delta 0), reused 3 (delta 0), pack-reused 2685 (from 2)
Receiving objects: 100% (2691/2691), 904.02 MiB | 40.42 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Updating files: 100% (2000/2000), done.
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [10]:
cd sem-fracture-deep-learning

/kaggle/working/sem-fracture-deep-learning


In [12]:
!git checkout vishnu-dev
!pip install -r requirements.txt

Updating files: 100% (635/635), done.
branch 'vishnu-dev' set up to track 'origin/vishnu-dev'.
Switched to a new branch 'vishnu-dev'


# Fracture modelling

Use the sections below in order: Train, Validate, then optionally Test. Mixed augmentation is training-only. Final test evaluation is disabled until explicitly enabled. The stored training output is from an earlier interrupted attempt.


## Train

Configure the model in the setup cell and run training. The best validation-macro-F1 checkpoint is saved under `results/shared_pipeline/`.


In [22]:
from pathlib import Path

import pandas as pd
import torch

from sem_fracture.pipeline import run_experiment, run_prediction

DATA_DIR = Path.cwd()
MODEL_KEY = "tinyvit_21m_dual_attention"  # resnet18, deit_tiny, tinyvit_11m, tinyvit_21m, tinyvit_21m_dual_attention, dinov2_vitb14
MODE = "finetune"  # head_only or finetune
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

BASELINE_CHECKPOINTS = {
    "resnet18": DATA_DIR / "results" / "resnet18_best.pth",
    "deit_tiny": DATA_DIR / "results" / "deit_tiny" / "checkpoints" / "deit_tiny_best.pth",
    "tinyvit_11m": DATA_DIR / "results" / "tinyvit_11m" / "checkpoints" / "tinyvit_11m_best.pth",
    "tinyvit_21m": DATA_DIR / "results" / "tinyvit_21m" / "checkpoints" / "tinyvit_21m_best.pth",
    "tinyvit_21m_dual_attention": DATA_DIR / "results" / "tinyvit_21m_finetune" / "checkpoints" / "tinyvit_21m_finetuned_best.pth",
    "dinov2_vitb14": DATA_DIR / "results" / "shared_pipeline" / "dinov2_vitb14" / "dinov2_vitb14_head_only_best.pth",
}

if MODE not in ("head_only", "finetune"):
    raise ValueError(f"Unknown training mode: {MODE}")
if MODEL_KEY not in BASELINE_CHECKPOINTS:
    raise ValueError(f"Unknown model key: {MODEL_KEY}")

IS_DUAL_ATTENTION = MODEL_KEY == "tinyvit_21m_dual_attention"
_baseline = BASELINE_CHECKPOINTS[MODEL_KEY]
if (
    MODEL_KEY == "dinov2_vitb14"
    and isinstance(_baseline, Path)
    and not _baseline.is_file()
):
    INITIAL_CHECKPOINT = None
    print(
        "No DINOv2 head-only checkpoint on disk. "
        "Starting from the pretrained torch.hub weights."
    )
else:
    INITIAL_CHECKPOINT = (
        _baseline if (MODE == "finetune" or IS_DUAL_ATTENTION) else None
    )
BACKBONE_LR = 1e-5 if (MODE == "finetune" and not IS_DUAL_ATTENTION) else None
HEAD_LR = 1e-4 if (MODE == "finetune" or IS_DUAL_ATTENTION) else 1e-3
EPOCHS = 30 if (MODEL_KEY.startswith("tinyvit_21m") and (MODE == "finetune" or IS_DUAL_ATTENTION)) else 10
PATIENCE = 4 if (MODE == "finetune" or IS_DUAL_ATTENTION) else None
GRAD_CLIP_NORM = 1.0 if (MODEL_KEY.startswith("tinyvit") and MODE == "finetune" and not IS_DUAL_ATTENTION) else None
AUGMENT_MIXED = False if IS_DUAL_ATTENTION else True
CHECKPOINT_PATH = (
    DATA_DIR / "results" / "shared_pipeline"
    / MODEL_KEY / f"{MODEL_KEY}_{MODE}_best.pth"
)
BATCH_SIZE = 8 if MODEL_KEY == "dinov2_vitb14" else 16
LOSS_TYPE = "cross_entropy"  # cross_entropy, binary_cross_entropy, class_balanced
RUN_FINAL_TEST_EVALUATION = False  # enable only after selecting the checkpoint


In [23]:
result = run_experiment(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    mode=MODE,
    checkpoint_path=CHECKPOINT_PATH,
    device=DEVICE,
    head_lr=HEAD_LR,
    backbone_lr=BACKBONE_LR,
    weight_decay=1e-4,
    epochs=EPOCHS,
    patience=PATIENCE,
    grad_clip_norm=GRAD_CLIP_NORM,
    initial_checkpoint=INITIAL_CHECKPOINT,
    augment_mixed=AUGMENT_MIXED,
    seed=42,
    batch_size=BATCH_SIZE,
    loss_type=LOSS_TYPE,
)

best = result["best_checkpoint"]
print("Best epoch:", best["epoch"])
print("Best validation macro-F1:", f"{best['val_macro_f1']:.4f}")
print("Checkpoint:", CHECKPOINT_PATH)
print("Training split:", result["split_file"])
print("Class weights:", result["class_weights"])


Epoch 1/30 starting
Epoch 1/30 | train_loss=0.1710 | val_loss=0.1049 | val_accuracy=0.9571 | val_macro_f1=0.9079 | elapsed=30.1s | best_checkpoint=True
Epoch 2/30 starting
Epoch 2/30 | train_loss=0.1115 | val_loss=0.1362 | val_accuracy=0.9464 | val_macro_f1=0.8934 | elapsed=27.5s | best_checkpoint=False
Epoch 3/30 starting
Epoch 3/30 | train_loss=0.0994 | val_loss=0.0727 | val_accuracy=0.9643 | val_macro_f1=0.9213 | elapsed=27.9s | best_checkpoint=True
Epoch 4/30 starting
Epoch 4/30 | train_loss=0.0761 | val_loss=0.0983 | val_accuracy=0.9571 | val_macro_f1=0.9079 | elapsed=27.9s | best_checkpoint=False
Epoch 5/30 starting
Epoch 5/30 | train_loss=0.0665 | val_loss=0.0509 | val_accuracy=0.9786 | val_macro_f1=0.9475 | elapsed=28.0s | best_checkpoint=True
Epoch 6/30 starting
Epoch 6/30 | train_loss=0.0501 | val_loss=0.0692 | val_accuracy=0.9750 | val_macro_f1=0.9420 | elapsed=27.7s | best_checkpoint=False
Epoch 7/30 starting
Epoch 7/30 | train_loss=0.0411 | val_loss=0.0960 | val_accuracy=0

## Validate

Evaluate the selected checkpoint on the original validation split. Validation data is not augmented.


In [24]:
validation_result = run_prediction(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    checkpoint_path=CHECKPOINT_PATH,
    split="validation",
    split_file="dataset_split_augmented.csv",
    device=DEVICE,
    batch_size=BATCH_SIZE,
)

print(validation_result["metrics"])
pd.DataFrame(validation_result["predictions"])


{'confusion_matrix': [[117, 0, 0], [0, 138, 4], [0, 2, 19]], 'accuracy': 0.9785714285714285, 'precision': [1.0, 0.9857142857142858, 0.8260869565217391], 'recall': [1.0, 0.971830985915493, 0.9047619047619048], 'class_f1': [1.0, 0.9787234042553192, 0.8636363636363636], 'macro_f1': 0.9474532559638943}


,filename,actual,predicted
0,100Cr6_AR_06_10.png,Brittle,Brittle
1,100Cr6_AR_06_3.png,Brittle,Brittle
2,304_H2_02_1.png,Brittle,Brittle
3,304_H2_02_10.png,Brittle,Brittle
4,304_H2_02_11.png,Brittle,Brittle
...,...,...,...
275,mX65_H2_29_22.png,Mixed,Mixed
276,mX65_H2_29_23.png,Mixed,Mixed
277,mX65_H2_29_3.png,Mixed,Mixed
278,mX65_H2_29_31.png,Mixed,Mixed


## Test

After selecting the checkpoint with validation, set `RUN_FINAL_TEST_EVALUATION = True` in the setup cell and run the cell below once. It uses the original `dataset_split.csv`; generated Mixed images belong to training only.


In [25]:
RUN_FINAL_TEST_EVALUATION = True

In [26]:
if RUN_FINAL_TEST_EVALUATION:
    final_test_result = run_prediction(
        data_dir=DATA_DIR,
        model_key=MODEL_KEY,
        checkpoint_path=CHECKPOINT_PATH,
        split="test",
        split_file="dataset_split_augment.csv",
        device=DEVICE,
        batch_size=BATCH_SIZE,
    )
    print("Final held-out test metrics:", final_test_result["metrics"])
else:
    print("Final test evaluation is disabled.")


Final held-out test metrics: {'confusion_matrix': [[116, 0, 0], [0, 140, 3], [4, 2, 14]], 'accuracy': 0.967741935483871, 'precision': [0.9666666666666667, 0.9859154929577465, 0.8235294117647058], 'recall': [1.0, 0.9790209790209791, 0.7], 'class_f1': [0.983050847457627, 0.9824561403508772, 0.7567567567567567], 'macro_f1': 0.9074212481884203}
